# ⚡ LMTrade on Google Colab

Run the self-sustaining hybrid LLM/SLM trading bot in Colab. Colab gives you a free GPU to serve the SLM, but note:

- **Paper mode** is the default — no real money, no keys required.
- The runtime is **ephemeral**: state resets on disconnect unless you mount Drive (cell 3).
- Colab kills idle/long runtimes (~12h), so this is for **testing/experimentation**, not an always-on bot (use the Vast.ai path for that).
- Colab can't expose ports directly, so the dashboard is served through Colab's **port proxy** (last cell).

Run the cells top to bottom.

## 1. Install LMTrade

In [ ]:
!git clone https://github.com/269652/LMTrade.git
%cd LMTrade
!pip install -e . -q
print('\n✅ LMTrade installed')

## 2. (Optional) Serve a local SLM with Ollama on the Colab GPU

Skip this cell to run with the free **heuristic** financial model only. Enable `Runtime → Change runtime type → GPU` first for decent SLM speed.

In [ ]:
import os, subprocess, time

SLM_MODEL = 'qwen2.5:1.5b'  # small, fits a free T4

!curl -fsSL https://ollama.com/install.sh | sh
subprocess.Popen(['ollama', 'serve'])
time.sleep(5)
!ollama pull {SLM_MODEL}

os.environ['OLLAMA_HOST'] = 'http://localhost:11434'
os.environ['LMTRADE_SLM_MODEL'] = SLM_MODEL
os.environ['LMTRADE_MODEL_STACK'] = 'heuristic,slm'
print('\n✅ Ollama serving', SLM_MODEL)

## 3. (Optional) Persist state to Google Drive

Without this, your trade history / equity curve is lost when the runtime disconnects. This points LMTrade's `data/` directory at a folder in your Drive so history survives across sessions.

In [ ]:
PERSIST_TO_DRIVE = True  # set False to keep state ephemeral

if PERSIST_TO_DRIVE:
    from google.colab import drive
    import os, pathlib
    drive.mount('/content/drive')
    data_dir = '/content/drive/MyDrive/lmtrade_data'
    pathlib.Path(data_dir).mkdir(parents=True, exist_ok=True)
    # Symlink the repo's data/ dir to Drive so the SQLite store lives there.
    if os.path.islink('data') or os.path.exists('data'):
        os.path.islink('data') and os.unlink('data')
    if not os.path.exists('data'):
        os.symlink(data_dir, 'data')
    print('✅ State persists to', data_dir)
else:
    print('State is ephemeral (will reset on disconnect).')

## 4. Configure

Paper mode, €10 budget. GPU rate is $0 on the free tier (Colab Pro is ~$0.15/hr — change if relevant). Add API keys here if you have them.

In [ ]:
import os

os.environ['LMTRADE_MODE'] = 'paper'
os.environ['LMTRADE_BUDGET'] = '10'
os.environ['LMTRADE_UNIVERSE'] = 'AAPL,MSFT,SPY'
os.environ['LMTRADE_GPU_USD_PER_HOUR'] = '0'   # free tier; Colab Pro ~0.15

# Optional keys — uncomment and fill in to enable cloud LLM / Perplexity research:
# os.environ['ANTHROPIC_API_KEY'] = '...'
# os.environ['PERPLEXITY_API_KEY'] = '...'
# os.environ['LMTRADE_MODEL_STACK'] = 'heuristic,slm,perplexity'

# Use live market data (yfinance) instead of the synthetic fallback:
!pip install -q yfinance
print('✅ Configured:', os.environ.get('LMTRADE_MODEL_STACK', 'heuristic'))

## 5. Quick check — run a few cycles inline

Sanity-run the engine for 3 cycles so you can see decisions/trades in the output before starting the background loop.

In [ ]:
!lmtrade run --cycles 3 --interval 1
print('\n--- status ---')
!lmtrade status

## 6. Start the engine in the background

Colab cells block, so we launch the continuous loop as a background process. It keeps trading while you view the dashboard.

In [ ]:
import subprocess

engine = subprocess.Popen(['lmtrade', 'run', '--interval', '30'])
print('✅ Engine running in background (pid', engine.pid, '). Evaluates every 30s.')

## 7. Launch the dashboard (via Colab port proxy)

Click the printed URL to open the live dashboard: portfolio, economics / self-sustaining status, trades, activity feed, logs and the equity curve.

In [ ]:
import subprocess, time

web = subprocess.Popen(['lmtrade', 'web', '--host', '0.0.0.0', '--port', '8000'])
time.sleep(5)
from google.colab.output import eval_js
url = eval_js('google.colab.kernel.proxyPort(8000)')
print('📊 Dashboard →', url)

## Stop everything

Run this to terminate the background engine and web server.

In [ ]:
for p in ('engine', 'web'):
    proc = globals().get(p)
    if proc is not None:
        proc.terminate()
        print('stopped', p)